# Member Analytics Report

Season snapshots, season-to-season trends, and season-over-season retention for Team Rynkeby Hamburg, built from every season already scraped and (optionally) geocoded. Read-only: this notebook never geocodes and never writes to any season record. Run every cell top to bottom (**Run All**).

See `specs/005-member-analytics-report/` for the full design, and `specs/005-member-analytics-report/contracts/cli-and-env.md` for how to run and export this notebook.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

if "scripts" not in sys.modules and str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

raw_data_dir = os.environ.get("RKBY_DATA_DIR")
if not raw_data_dir:
    raise RuntimeError("Missing required environment variable: RKBY_DATA_DIR")

RKBY_DATA_DIR = Path(raw_data_dir)
if not RKBY_DATA_DIR.is_dir():
    raise RuntimeError(
        f"RKBY_DATA_DIR does not exist or is not a directory: {RKBY_DATA_DIR}"
    )

RKBY_DATA_DIR

In [ ]:
from scripts.rkby_report.frame import ensure_reports_dir_and_gitignore

ensure_reports_dir_and_gitignore(RKBY_DATA_DIR)

## Season snapshot (User Story 1)

Role, gender, age, and distance-from-Hamburg breakdown for one season, defaulting to the most recently discovered season. Members missing a field show up under an explicit "unknown" category rather than disappearing from the count.

In [ ]:
from scripts.rkby_records import discover_seasons
from scripts.rkby_report import plots
from scripts.rkby_report.aggregate import data_gaps, season_summary
from scripts.rkby_report.frame import build_member_season_frame

member_season_frame = build_member_season_frame(RKBY_DATA_DIR)
discovered_seasons = discover_seasons(RKBY_DATA_DIR)
selected_season = discovered_seasons[-1] if discovered_seasons else None
selected_season

In [ ]:
summary = season_summary(member_season_frame, selected_season)
plots.role_chart(summary)
plots.gender_chart(summary)
plots.age_bucket_chart(summary)
plots.distance_bucket_chart(summary)

In [ ]:
# Maintainer-only: which members are missing a field a view needs
# (FR-017), by match_key -- never name/address/phone/birthday. Tagged
# 'remove-cell' so nbconvert's --TagRemovePreprocessor strips this cell
# (source and output) from the FR-014 export (research.md §11).
data_gaps(member_season_frame)

## Season-to-season trends (User Story 2)

Total/rider/service-crew counts and age/gender/distance distribution shifts across every discovered season. With only one season on file, these charts say plainly that there isn't enough data yet instead of plotting a misleading single point.

In [ ]:
from scripts.rkby_report.aggregate import season_trend

trend = season_trend(member_season_frame)
plots.member_count_trend_chart(trend)
plots.age_distribution_shift_chart(trend)
plots.gender_distribution_shift_chart(trend)
plots.distance_distribution_shift_chart(trend)

## Retention (User Story 3)

Season-over-season retention -- overall, and split by gender, age bracket, and distance-from-Hamburg bracket -- for every consecutive pair of discovered seasons.

In [ ]:
import itertools

from scripts.rkby_report.aggregate import retention_by_split, retention_cohort

for season_a, season_b in itertools.pairwise(discovered_seasons):
    cohort = retention_cohort(member_season_frame, season_a, season_b)
    plots.overall_retention_chart(cohort)
    plots.retention_by_split_chart(
        retention_by_split(member_season_frame, season_a, season_b, "sex"),
        "gender",
    )
    plots.retention_by_split_chart(
        retention_by_split(member_season_frame, season_a, season_b, "age_bucket"),
        "age bracket",
    )
    plots.retention_by_split_chart(
        retention_by_split(member_season_frame, season_a, season_b, "distance_bucket"),
        "distance bracket",
    )

## Sharing the finished report (User Story 4)

Export every chart and summary table above to one self-contained HTML file under `$RKBY_DATA_DIR/reports/` -- aggregates only, no per-member roster, and never the data-gap list above (its `remove-cell` tag is what strips it here):

```bash
uv run jupyter nbconvert --to html --execute \
  --TagRemovePreprocessor.remove_cell_tags='{"remove-cell"}' \
  scripts/report_member_analytics.ipynb \
  --output-dir "$RKBY_DATA_DIR/reports"
```

See `specs/005-member-analytics-report/contracts/cli-and-env.md` for the full contract.